# Sheikh Hamdan Speaker Recognition
This notebook loops through audio files, removes pauses, takes the first 5–10 seconds of speech, and identifies if the speaker is Sheikh Hamdan bin Mohammed.

In [ ]:
# Install required packages
!pip install torch torchaudio pyannote.audio librosa

In [ ]:
import os
import torch
import librosa
import numpy as np
from pyannote.audio import Inference
from torch.nn.functional import cosine_similarity
import json

## Step 1: Define function to remove silence and extract first 5–10s

In [ ]:
def extract_speech_segment(path, target_seconds=8, sr=16000):
    y, _ = librosa.load(path, sr=sr)
    intervals = librosa.effects.split(y, top_db=25)
    speech = np.concatenate([y[start:end] for start, end in intervals])
    max_len = target_seconds * sr
    return speech[:max_len]

## Step 2: Load speaker embedding model

In [ ]:
speaker_model = Inference(
    "pyannote/embedding",
    window="whole"
)

## Step 3: Build Sheikh Hamdan reference embedding

In [ ]:
reference_files = [
    "hamdan_1.wav",
    "hamdan_2.wav",
    "hamdan_3.wav",
]

ref_embeddings = []
for f in reference_files:
    speech = extract_speech_segment(f)
    emb = speaker_model({"waveform": torch.tensor(speech).unsqueeze(0), "sample_rate": 16000})
    ref_embeddings.append(emb)

reference_embedding = torch.mean(torch.stack(ref_embeddings), dim=0)

## Step 4: Define function to check if speaker matches Sheikh Hamdan

In [ ]:
def is_sheikh_hamdan(audio_path, threshold=0.75):
    speech = extract_speech_segment(audio_path)
    emb = speaker_model({"waveform": torch.tensor(speech).unsqueeze(0), "sample_rate": 16000})
    score = cosine_similarity(emb, reference_embedding).item()
    return score > threshold, score

## Step 5: Loop through audio files and annotate results

In [ ]:
audio_dir = "audios"  # Folder containing your 200 audio files
results = []

for file in os.listdir(audio_dir):
    if not file.endswith(".wav"):
        continue

    is_match, score = is_sheikh_hamdan(os.path.join(audio_dir, file))
    results.append({
        "audio": file,
        "recognize": is_match,
        "confidence": round(score, 3)
    })

print(json.dumps(results, indent=2, ensure_ascii=False))

### ? Notes
- `target_seconds=8` controls how many seconds of speech are used.
- `threshold=0.75` controls strictness for matching Sheikh Hamdan.
- Silence is removed using `librosa.effects.split`.
- You can adjust `top_db` to be more or less aggressive in trimming pauses.